In [ ]:
import gensim
import pandas as pd
import re
import nltk
from gensim.models import word2vec,KeyedVectors

In [ ]:
import gensim.downloader as api

wv=api.load('word2vec-google-news-300')


In [134]:
vec_king=wv['king']

In [135]:
vec_king.shape

(300,)

In [136]:
messages = pd.read_csv('SpamClassifier-master\\smsspamcollection\\SMSSpamCollection', sep='\t', names=["label", "message"])

In [137]:
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()


In [138]:
corpus=[]
for i in range(0,len(messages)):
    review=re.sub('[^a-zA-Z]',' ',messages['message'][i])
    review=review.lower()
    review=review.split()
    review=[lemmatizer.lemmatize(word) for word in review]
    review=' '.join(review)
    corpus.append(review)

In [ ]:
corpus

In [139]:
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

In [140]:
words = []
for sent in corpus:
    sent_token = sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [ ]:
## lets train word2vec from scratch
model=gensim.models.Word2Vec(words)

In [142]:
## get all the vocabulary
model.wv.index_to_key


['to',
 'you',
 'the',
 'it',
 'and',
 'in',
 'is',
 'me',
 'my',
 'for',
 'your',
 'call',
 'of',
 'that',
 'have',
 'on',
 'now',
 'are',
 'can',
 'so',
 'but',
 'not',
 'or',
 'we',
 'do',
 'get',
 'at',
 'ur',
 'will',
 'if',
 'be',
 'with',
 'no',
 'just',
 'this',
 'gt',
 'lt',
 'go',
 'how',
 'up',
 'when',
 'ok',
 'day',
 'what',
 'free',
 'from',
 'all',
 'out',
 'know',
 'll',
 'come',
 'like',
 'good',
 'time',
 'am',
 'then',
 'got',
 'wa',
 'there',
 'he',
 'love',
 'text',
 'only',
 'want',
 'send',
 'one',
 'need',
 'txt',
 'today',
 'by',
 'going',
 'don',
 'stop',
 'home',
 'she',
 'about',
 'lor',
 'sorry',
 'see',
 'still',
 'mobile',
 'take',
 'back',
 'da',
 'reply',
 'dont',
 'our',
 'think',
 'tell',
 'week',
 'hi',
 'phone',
 'they',
 'new',
 'please',
 'later',
 'pls',
 'any',
 'her',
 'ha',
 'co',
 'did',
 'been',
 'msg',
 'min',
 'some',
 'an',
 'night',
 'make',
 'dear',
 'who',
 'here',
 'message',
 'say',
 'well',
 'where',
 're',
 'thing',
 'much',
 'oh',

In [143]:
model.corpus_count

5569

In [144]:
model.wv.similar_by_word('good')

[('my', 0.9991218447685242),
 ('all', 0.99893718957901),
 ('well', 0.9988796710968018),
 ('day', 0.9988337159156799),
 ('dear', 0.9987387657165527),
 ('morning', 0.9987227916717529),
 ('having', 0.9986761808395386),
 ('night', 0.9986503720283508),
 ('thing', 0.9986480474472046),
 ('wa', 0.998613178730011)]

In [145]:
model.wv['good'].shape


(100,)

In [158]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [159]:
from tqdm import tqdm

In [160]:
import numpy as np

In [252]:
def avg_word2vec(doc):
    
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)
    
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)
                #or [np.zeros(len(model.wv.index_to_key))], axis=0)

In [253]:
y = messages[list(map(lambda x: len(x)>0 ,corpus))]
y=pd.get_dummies(y['label'])
y=y.iloc[:,0].values

In [254]:
## apply for entire sentences
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))



  0%|          | 0/5569 [00:00<?, ?it/s]c:\Users\sarth\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\sarth\anaconda3\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 5569/5569 [00:00<00:00, 8875.13it/s] 


In [255]:
# 1. Create an empty list to store your rows
data_list = []

for i in range(0, len(X)):
    # 2. Append the new data to the LIST, not the DataFrame
    data_list.append(pd.DataFrame(X[i].reshape(1, -1)))

# 3. Combine everything into one DataFrame at the end
df = pd.concat(data_list, ignore_index=True)

C:\Users\sarth\AppData\Local\Temp\ipykernel_9520\3246507674.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(data_list, ignore_index=True)


In [256]:
df['100'] = y

In [257]:
df.dropna(inplace=True)

In [258]:
y = df['100']

In [259]:
X = df

In [260]:
## this was supposed to be written before the train test split
X.columns = X.columns.astype(str)

In [261]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.5)

In [262]:
y_train = y_train.astype(int)

In [207]:
y_train

5552    1
3968    1
4888    1
3306    1
1601    1
       ..
5495    1
3496    1
4306    1
1963    1
2046    1
Name: 100, Length: 4445, dtype: int32

In [247]:
from sklearn.ensemble import RandomForestClassifier
classfier = RandomForestClassifier()

In [217]:
y_train

810     1
3524    1
5489    0
3603    1
3663    1
       ..
1317    1
4348    1
4881    1
4719    1
4060    1
Name: 100, Length: 4445, dtype: int32

In [263]:
classfier.fit(X_train,y_train)

RandomForestClassifier()

In [264]:
y_pred=classfier.predict(X_test)

In [242]:
from sklearn.metrics import accuracy_score,classification_report

In [265]:
accuracy_score(y_test,y_pred)

0.9953220582943505

In [266]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.99      0.97      0.98       364
        True       1.00      1.00      1.00      2415

    accuracy                           1.00      2779
   macro avg       0.99      0.98      0.99      2779
weighted avg       1.00      1.00      1.00      2779

